In [1]:
!pip install langchain langchain-community langchain-core langchain-groq faiss-cpu sentence-transformers beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [6]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq

def load_website(url):
    loader = WebBaseLoader(url)
    return loader.load()


def split_docs(docs):
    splitter = CharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    return splitter.split_documents(docs)


def create_db(chunks):
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2"
    )
    return FAISS.from_documents(chunks, embeddings)



def get_retriever(db):
    return db.as_retriever(search_kwargs={"k": 4})



def load_llm():
    return ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0
    )



def build_chain(retriever, llm):

    prompt = ChatPromptTemplate.from_template(
        """Answer ONLY from the website content below.
If answer not found, say "Not available on website".

Context:
{context}

Question:
{question}

Answer:"""
    )

    chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
    )

    return chain


def main():

    url = input("Enter website URL: ")

    print("\nLoading website...\n")

    docs = load_website(url)
    chunks = split_docs(docs)
    db = create_db(chunks)
    retriever = get_retriever(db)
    llm = load_llm()
    chain = build_chain(retriever, llm)

    print("\n===== WEBSITE BOT READY =====\n")

    while True:
        query = input("Ask question: ")

        if query.lower() == "exit":
            break

        response = chain.invoke(query)

        print("\nAnswer:\n", response.content)
        print("\n" + "="*50)



if __name__ == "__main__":
    main()

Enter website URL: https://www.ibm.com/think/topics/neural-networks

Loading website...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== WEBSITE BOT READY =====

Ask question: what is neural network

Answer:
 A neural network is a machine learning model that stacks simple "neurons" in layers and learns pattern-recognizing weights and biases from data to map inputs to outputs.

Ask question: what is artificial intelligence

Answer:
 Not available on website

Ask question: How do neural network works

Answer:
 A neural network is comprised of:

Input layer: holds the raw features (X1,X2,X3,...)

Hidden layers: consist of artificial neurons (or nodes) that transform inputs into new representations. Mathematically, hidden layers are expressed as the input features, multiplied by their associated weights and added bias to pass from one layer to the next layer, eventually arriving at the final output layer. This is where the linear transformation between input and output happens.

Neural network training requires rigorous training to perform well on testing. To train a network, a single neuron computes:

z=∑i=1nwixi+b
